#### EDA 09: Category translation table
**File:** `product_category_name_translation.csv`  |  **Goal:** understand the lookup from Portuguese category names to English names, using this table only.

#### Load and first look
**Question:** how big is the table and what does a row look like?

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

RAW = Path("../../data/raw")                      # path from the notebook folder to the raw data
cat_en_raw = pd.read_csv(RAW / "product_category_name_translation.csv")
cat_en = cat_en_raw.copy()                        # all changes happen on the copy, raw stays untouched

print("shape:", cat_en.shape)
print(cat_en.dtypes)
cat_en.head(5)

shape: (71, 2)
product_category_name            str
product_category_name_english    str
dtype: object


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


- 71 rows and 2 columns: the Portuguese category name and its English name. Both are text.
- It is a very small lookup table. One row is one category.

#### Keys, duplicates and missing values
**Question:** is the Portuguese name a valid key, and are there gaps?

In [2]:
pt = cat_en["product_category_name"]
en = cat_en["product_category_name_english"]

print("fully duplicated rows              :", cat_en.duplicated().sum())
print("Portuguese name unique             :", pt.is_unique)
print("English name unique                :", en.is_unique)
print("two Portuguese names, one English  :", en.duplicated().sum())

# Column-wise missing count and percentage, for every column (zeros included)
missing = pd.DataFrame({"n_rows": len(cat_en),
                        "n_missing": cat_en.isna().sum(),
                        "pct_missing": (cat_en.isna().mean() * 100).round(2)})
display(missing.sort_values("n_missing", ascending=False))

fully duplicated rows              : 0
Portuguese name unique             : True
English name unique                : True
two Portuguese names, one English  : 0


,n_rows,n_missing,pct_missing
product_category_name,71,0,0.0
product_category_name_english,71,0,0.0


- No duplicated rows and no missing values.
- Both the Portuguese and the English name are unique, so the table is a clean one-to-one lookup. The Portuguese name is the key (it is the name used in the products table).

#### Names that are the same in both languages
**Question:** which names are unchanged by the translation?

In [3]:
same = cat_en[pt == en]
print("names identical in both languages:", len(same))
same

names identical in both languages: 7


,product_category_name,product_category_name_english
19,consoles_games,consoles_games
20,audio,audio
22,cool_stuff,cool_stuff
29,pet_shop,pet_shop
31,market_place,market_place
60,la_cuisine,la_cuisine
65,dvds_blu_ray,dvds_blu_ray


- 7 names are identical: `consoles_games`, `audio`, `cool_stuff`, `pet_shop`, `market_place`, `la_cuisine`, `dvds_blu_ray`.
- They are English or foreign words that Portuguese already uses, so this is not a translation gap.

#### Spelling in the English names
**Question:** are there spelling errors in the English names?

*What it does:* compares every word of the English names with every other word and lists pairs that differ by exactly one letter. Such pairs are either two real word forms or a typo.

In [4]:
def edit_distance(a, b):
    """Number of single-letter changes needed to turn a into b."""
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]


def close_word_pairs(names):
    counts = pd.Series([w for n in names for w in n.split("_")]).value_counts()
    words = counts.index.tolist()
    rows = []
    for i, a in enumerate(words):
        for b in words[i + 1:]:
            if len(a) >= 4 and len(b) >= 4 and edit_distance(a, b) == 1:
                rows.append({"word_a": a, "count_a": counts[a], "word_b": b, "count_b": counts[b]})
    return pd.DataFrame(rows)


print("word pairs that differ by one letter (English names)")
display(close_word_pairs(en))

# Names found this way, plus one found by reading the table (craftmanship)
EN_FIXES = {
    "fashio_female_clothing": "fashion_female_clothing",
    "costruction_tools_garden": "construction_tools_garden",
    "costruction_tools_tools": "construction_tools_tools",
    "home_confort": "home_comfort",
    "arts_and_craftmanship": "arts_and_craftsmanship",
}
print("names with a spelling error")
display(cat_en[en.isin(EN_FIXES)].assign(suggested=lambda d: d["product_category_name_english"].map(EN_FIXES)))

print("suspected wrong translation (seguros normally means insurance)")
display(cat_en[pt == "seguros_e_servicos"])

word pairs that differ by one letter (English names)


,word_a,count_a,word_b,count_b
0,fashion,6,fashio,1
1,construction,5,costruction,2
2,sports,1,sport,1
3,drink,1,drinks,1
4,confort,1,comfort,1
5,musical,1,musicals,1


names with a spelling error


,product_category_name,product_category_name_english,suggested
27,construcao_ferramentas_jardim,costruction_tools_garden,construction_tools_garden
35,casa_conforto,home_confort,home_comfort
36,construcao_ferramentas_ferramentas,costruction_tools_tools,construction_tools_tools
55,fashion_roupa_feminina,fashio_female_clothing,fashion_female_clothing
67,artes_e_artesanato,arts_and_craftmanship,arts_and_craftsmanship


suspected wrong translation (seguros normally means insurance)


,product_category_name,product_category_name_english
70,seguros_e_servicos,security_and_services


- The check finds 6 pairs. Three are real word forms and not errors: `sports` / `sport`, `drink` / `drinks` and `musical` / `musicals`.
- Three are spelling errors: `fashio` (for `fashion`), `costruction` (for `construction`, used in 2 names) and `confort` (for `comfort`). This affects 4 names.
- One more error was found by reading the table: `arts_and_craftmanship` (should be `craftsmanship`). The check cannot find it, because the correct word appears nowhere else.
- `home_confort` and `home_comfort_2` are the same pair in Portuguese (`casa_conforto` and `casa_conforto_2`), but only the first has the typo.
- Suspected translation error: `seguros_e_servicos` is translated as `security_and_services`, but `seguros` normally means insurance. It is flagged and not changed, because the intended meaning is not in the data.

In [5]:
# Explicit correction list (5 names). The raw column is untouched; a clean column is added.
cat_en["category_en_clean"] = en.replace(EN_FIXES)

print("names changed       :", (cat_en["product_category_name_english"] != cat_en["category_en_clean"]).sum())
print("clean names unique  :", cat_en["category_en_clean"].is_unique)
print("\nword pairs that differ by one letter, after the fix")
display(close_word_pairs(cat_en["category_en_clean"]))

fam_clean = cat_en["category_en_clean"].str.split("_").str[0].value_counts()
print("English first words after the fix:", fam_clean[fam_clean >= 3].to_dict())

names changed       : 5
clean names unique  : True

word pairs that differ by one letter, after the fix


,word_a,count_a,word_b,count_b
0,sports,1,sport,1
1,drink,1,drinks,1
2,musical,1,musicals,1


English first words after the fix: {'fashion': 7, 'construction': 5, 'home': 5, 'furniture': 4, 'books': 3}


- 5 names are corrected and the clean names are still unique.
- The one-letter pairs left are the 3 real word forms (`sports` / `sport`, `drink` / `drinks`, `musical` / `musicals`).
- After the fix the English families match the Portuguese ones: `fashion` has 7 names and `construction` 5.
- **Decision:** keep the raw English column untouched, add `category_en_clean` with the 5 explicit corrections, and use the clean column for display. Use the Portuguese name as the key. The suspected mistranslation (`seguros_e_servicos`) is flagged, not changed.